# FLAMES: brightest Gaussian component per spectrum
Run from the top after setting `out_dir` to the desired saved run (two or five components). No refitting is performed.

**Brightest means largest finite positive fitted peak amplitude**, not integrated Gaussian flux. Ties select the lowest original component index. Selection is not a quality cut or a physical association across sightlines.

- `df_validation_summary`: one row per input spectrum, with arrays of component parameters.
- `df_components`: one row per Gaussian, with scalar component parameters.
- `df_representative`: one row per input spectrum; selected Gaussian parameters are scalar. Missing representatives retain NaNs and an explicit status.
- `df_plot`: representatives with finite positive widths and finite velocities, used for the following kinematic plots. Fit-quality labels remain available but are not automatically excluded.

The count map uses the full summary, including zero-component fits. Exports go into `out_dir / "brightest_component"`.

In [ ]:
import pickle
from pathlib import Path

out_dir = Path("/Users/javier/FLAMES_30dor/gausspyp_v02.2/gausspy_validation")

FILENAME_DATA = out_dir / "real_FLAMES_validation_input.pkl"
FILENAME_DATA_DECOMP = out_dir / "real_FLAMES_validation_decomposed.pkl"
FILENAME_DF_VALIDATION = out_dir / "df_validation.pkl"
FILENAME_CONFIG = out_dir / "validation_training_config.pkl"

with open(FILENAME_DATA, "rb") as f:
    gauss_input = pickle.load(f)

with open(FILENAME_DATA_DECOMP, "rb") as f:
    data_decomp = pickle.load(f)

with open(FILENAME_DF_VALIDATION, "rb") as f:
    df_validation = pickle.load(f)

with open(FILENAME_CONFIG, "rb") as f:
    training_config = pickle.load(f)

print("gauss_input:", type(gauss_input))
print("data_decomp:", type(data_decomp))
print("df_validation:", type(df_validation))
print("training_config:", training_config)

In [ ]:
# Shared conversion settings
LAMBDA0_NM = 656.279
VELOCITY_REFERENCE_KMS = 267.4
# Retain the two broadening terms from the original notebook.
# Their physical interpretation/calibration must be checked for your data.
BROADENING_1_KMS = 7.8
BROADENING_2_KMS = 9.1
FWHM_TO_SIGMA = (8 * __import__('math').log(2)) ** 0.5
COUNT_COLOR_MAX = 5  # fixed 0..5 colors for comparing runs; expands if needed

In [ ]:
import numpy as np
import pandas as pd

def as_vector(value):
    return np.array([], dtype=float) if value is None else np.asarray(value, dtype=float).reshape(-1)

def to_scalar(value, default=np.nan):
    """
    Convert GaussPy scalar-like outputs to a float.

    Handles cases like:
    - 1.23
    - [1.23]
    - np.array([1.23])
    - None
    """
    if value is None:
        return default

    arr = np.asarray(value)

    if arr.size == 0:
        return default

    return float(arr.ravel()[0])
import numpy as np
import pandas as pd


def make_validation_summary(
    gauss_input,
    data_decomp,
    lambda0_nm=LAMBDA0_NM,
    rchi2_threshold=20,
    many_components_threshold=5,
):
    """
    Build a compact validation summary table for GaussPy decompositions.

    One row corresponds to one analyzed spectrum.

    Parameters
    ----------
    gauss_input : dict
        Input dictionary used for GaussPy decomposition. Must contain:
        'data_list', 'x_values', 'errors', and 'metadata'.

    data_decomp : dict
        Output dictionary returned by GaussPy batch_decomposition.

    lambda0_nm : float, default=656.28
        Rest wavelength used to convert fitted centers into velocity.
        For Halpha, lambda0_nm ≈ 656.28 nm.

    rchi2_threshold : float, default=20
        Threshold above which the fit is flagged as large chi-square.

    many_components_threshold : int, default=5
        Threshold above which the fit is flagged as having many components.

    Returns
    -------
    summary : pandas.DataFrame
        Validation summary table.
    """

    c_kms = 299792.458

    rows = []

    n_spectra = len(gauss_input["data_list"])

    for i in range(n_spectra):

        metadata = gauss_input["metadata"][i]

        wave = np.asarray(metadata["wave"])

        if len(wave) > 1:
            dw = np.nanmedian(np.diff(wave))
        else:
            dw = np.nan

        wave0 = wave[0] if len(wave) > 0 else np.nan
        wave_min = np.nanmin(wave) if len(wave) > 0 else np.nan
        wave_max = np.nanmax(wave) if len(wave) > 0 else np.nan

        means_fit_chan = as_vector(data_decomp["means_fit"][i])
        fwhms_fit_chan = as_vector(data_decomp["fwhms_fit"][i])
        amps_fit = as_vector(data_decomp["amplitudes_fit"][i])

        means_initial = as_vector(data_decomp["means_initial"][i])
        fwhms_initial = as_vector(data_decomp["fwhms_initial"][i])
        amps_initial = as_vector(data_decomp["amplitudes_initial"][i])

        means_fit_err_chan = as_vector(data_decomp["means_fit_err"][i])
        fwhms_fit_err_chan = as_vector(data_decomp["fwhms_fit_err"][i])
        amps_fit_err = as_vector(data_decomp["amplitudes_fit_err"][i])

        ncomp_initial = len(means_initial)
        ncomp_fit = len(means_fit_chan)
        if len(amps_fit) != ncomp_fit or len(fwhms_fit_chan) != ncomp_fit:
            raise ValueError(f"Mismatched component arrays for input spectrum {i}")

        rchi2 = to_scalar(data_decomp["best_fit_rchi2"][i])

        # Convert channel quantities to wavelength quantities.
        means_fit_nm = wave0 + means_fit_chan * dw
        fwhms_fit_nm = fwhms_fit_chan * abs(dw)

        # Convert wavelength quantities to velocity quantities.
        means_fit_kms = c_kms * (means_fit_nm - lambda0_nm) / lambda0_nm
        fwhms_fit_kms = c_kms * fwhms_fit_nm / lambda0_nm

        # Compute sigma
        sigma_obs_kms = fwhms_fit_kms / FWHM_TO_SIGMA
        variance = sigma_obs_kms**2 - BROADENING_1_KMS**2 - BROADENING_2_KMS**2
        sigma_kms = np.full_like(variance, np.nan)
        valid_variance = np.isfinite(variance) & (variance >= 0) & (sigma_obs_kms > 0)
        sigma_kms[valid_variance] = np.sqrt(variance[valid_variance])

        # Basic automatic flags.
        flag_zero_components = ncomp_fit == 0
        flag_many_components = ncomp_fit > many_components_threshold
        flag_large_rchi2 = np.isfinite(rchi2) and rchi2 > rchi2_threshold

        if ncomp_fit > 0:
            flag_center_outside_window = np.any(
                (means_fit_nm < wave_min) | (means_fit_nm > wave_max)
            )
            flag_nonpositive_fwhm = np.any(fwhms_fit_chan <= 0)
        else:
            flag_center_outside_window = False
            flag_nonpositive_fwhm = False

        # Simple automatic quality label.
        if flag_zero_components:
            auto_quality = "zero_components"
        elif flag_nonpositive_fwhm:
            auto_quality = "nonpositive_fwhm"
        elif flag_center_outside_window:
            auto_quality = "center_outside_window"
        elif flag_large_rchi2:
            auto_quality = "large_rchi2"
        elif flag_many_components:
            auto_quality = "many_components"
        else:
            auto_quality = "ok_candidate"

        row = {
            # Identification
            "spectrum_index": i,
            "lambda0_nm": lambda0_nm,
            "validation_index": metadata.get("validation_index", i),
            "filename": metadata.get("filename", None),
            "object": metadata.get("object", None),
            "ra_deg": metadata.get("ra_deg", None),
            "dec_deg": metadata.get("dec_deg", None),
            "sample": metadata.get("sample", None),

            # Wavelength window
            "wave_min_nm": wave_min,
            "wave_max_nm": wave_max,
            "dw_nm": dw,

            # Component counts
            "ncomp_initial": ncomp_initial,
            "ncomp_fit": ncomp_fit,

            # Fit quality
            "best_fit_rchi2": rchi2,

            # Raw GaussPy outputs in channel units
            "means_initial_chan": means_initial,
            "fwhms_initial_chan": fwhms_initial,
            "amps_initial": amps_initial,

            "means_fit_chan": means_fit_chan,
            "fwhms_fit_chan": fwhms_fit_chan,
            "amps_fit": amps_fit,

            "means_fit_err_chan": means_fit_err_chan,
            "fwhms_fit_err_chan": fwhms_fit_err_chan,
            "amps_fit_err": amps_fit_err,

            # Converted quantities
            "means_fit_nm": means_fit_nm,
            "fwhms_fit_nm": fwhms_fit_nm,
            "means_fit_kms": means_fit_kms,
            "fwhms_fit_kms": fwhms_fit_kms,

            # Dispersion
            "sigma_obs_kms": sigma_obs_kms,
            "sigma_kms": sigma_kms,
            "flag_unresolved": (variance < 0),


            # Automatic flags
            "flag_zero_components": flag_zero_components,
            "flag_many_components": flag_many_components,
            "flag_large_rchi2": flag_large_rchi2,
            "flag_center_outside_window": flag_center_outside_window,
            "flag_nonpositive_fwhm": flag_nonpositive_fwhm,
            "auto_quality": auto_quality,

            # Manual validation fields
            "visual_quality": "",
            "notes": "",
        }

        rows.append(row)

    summary = pd.DataFrame(rows)

    return summary

In [ ]:
df_validation_summary = make_validation_summary(
    gauss_input,
    data_decomp,
    lambda0_nm=LAMBDA0_NM,
    rchi2_threshold=20,
    many_components_threshold=5,
)

display(df_validation_summary.head())

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

coords = SkyCoord(
    ra=df_validation_summary["ra_deg"].values * u.deg,
    dec=df_validation_summary["dec_deg"].values * u.deg,
    frame="icrs"
)

df_validation_summary["RA_hms"] = coords.ra.to_string(
    unit=u.hour,
    sep=":",
    precision=3,
    pad=True
)

df_validation_summary["Dec_dms"] = coords.dec.to_string(
    unit=u.deg,
    sep=":",
    precision=3,
    alwayssign=True,
    pad=True
)

In [ ]:
print(df_validation_summary["auto_quality"].value_counts())
print(df_validation_summary["ncomp_fit"].value_counts().sort_index())

In [ ]:
# Component arrays become scalar values in each component row.
IDENTITY_COLUMNS = ["spectrum_index", "validation_index", "filename", "object",
                    "ra_deg", "dec_deg", "ncomp_fit", "best_fit_rchi2",
                    "auto_quality", "visual_quality", "notes", "lambda0_nm",
                    "RA_hms", "Dec_dms"]
COMPONENT_COLUMNS = {
    "mean_nm": "means_fit_nm", "fwhm_nm": "fwhms_fit_nm",
    "mean_kms": "means_fit_kms", "fwhm_kms": "fwhms_fit_kms",
    "amp": "amps_fit", "amp_err": "amps_fit_err",
    "mean_err_chan": "means_fit_err_chan", "fwhm_err_chan": "fwhms_fit_err_chan",
    "sigma_obs_kms": "sigma_obs_kms", "sigma_kms": "sigma_kms",
    "flag_unresolved": "flag_unresolved",
}

def make_component_table(df_summary):
    records = []
    for _, row in df_summary.iterrows():
        for j in range(int(row["ncomp_fit"])):
            record = {key: row[key] for key in IDENTITY_COLUMNS}
            record["component_index"] = j
            for target, original in COMPONENT_COLUMNS.items():
                values = as_vector(row[original])
                record[target] = values[j] if j < len(values) else np.nan
            records.append(record)
    return pd.DataFrame(records, columns=IDENTITY_COLUMNS + ["component_index"] + list(COMPONENT_COLUMNS))


def select_brightest_components(df_summary, df_components):
    eligible = df_components.loc[
        np.isfinite(df_components["amp"].astype(float)) & (df_components["amp"] > 0)
    ]
    winners = (eligible.sort_values(
        ["spectrum_index", "amp", "component_index"], ascending=[True, False, True]
    ).drop_duplicates("spectrum_index"))
    selected_columns = ["spectrum_index", "component_index"] + list(COMPONENT_COLUMNS)
    result = df_summary[IDENTITY_COLUMNS].merge(
        winners[selected_columns], on="spectrum_index", how="left", validate="one_to_one"
    )
    result["component_index"] = result["component_index"].astype("Int64")
    result["has_representative"] = result["component_index"].notna()
    result["selection_status"] = np.where(
        result["has_representative"], "selected",
        np.where(result["ncomp_fit"].eq(0), "zero_components", "no_finite_positive_amplitude")
    )
    return result


df_components = make_component_table(df_validation_summary)
df_representative = select_brightest_components(df_validation_summary, df_components)
df_plot = df_representative.loc[
    df_representative["has_representative"]
    & np.isfinite(df_representative["mean_kms"])
    & np.isfinite(df_representative["fwhm_kms"])
    & (df_representative["fwhm_kms"] > 0)
].copy()

print(f"Input spectra: {len(df_validation_summary)}; fitted components: {len(df_components)}")
print(df_representative["selection_status"].value_counts())
print(f"Representatives with valid velocity and width: {len(df_plot)}")
display(df_representative.head())

## Spatial component counts
Each point is one input spectrum. Colors encode the actual fitted count, not the run's maximum. Zero-component spectra are included. The map is a decimal-degree coordinate overview.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

count_max = max(COUNT_COLOR_MAX, int(df_validation_summary["ncomp_fit"].max()))
count_cmap = plt.get_cmap("viridis", count_max + 1)
count_norm = BoundaryNorm(np.arange(-0.5, count_max + 1.5), count_cmap.N)
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
points = ax.scatter(
    df_validation_summary["ra_deg"], df_validation_summary["dec_deg"],
    c=df_validation_summary["ncomp_fit"], s=25,
    cmap=count_cmap, norm=count_norm,
)
ax.invert_xaxis()
ax.set(xlabel="RA (deg)", ylabel="Dec (deg)", title="Number of fitted Gaussian components")
ax.grid(alpha=0.2)
fig.colorbar(points, ax=ax, ticks=np.arange(count_max + 1), label="Fitted components")
plt.show()

## Representative-component diagnostics
All following histograms and maps use `df_plot`. The existing display limits are retained; histogram cuts are for display only. Unresolved corrected widths remain NaN and are omitted from the dispersion map. The reference velocity is a plotting offset, not a frame correction.

In [ ]:
import matplotlib.pyplot as plt

vel = df_plot["mean_kms"]

vel_filtered = vel[np.abs(vel) < 1000]

plt.figure(figsize=(7, 4))
plt.hist(vel_filtered, bins=100)
plt.xlabel("Component center velocity [km/s]")
plt.ylabel("Number of spectra")
plt.title("Velocity centers of brightest Gaussian components | |v| < 1000 km/s")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
fwhm = df_plot["fwhm_kms"]

fwhm_filtered = fwhm[(fwhm > 0) & (fwhm < 150)]

plt.figure(figsize=(7, 4))
plt.hist(fwhm_filtered, bins=100)
plt.xlabel("Component FWHM [km/s]")
plt.ylabel("Number of spectra")
plt.title("Velocity widths of brightest Gaussian components | 0 < FWHM < 150 km/s")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)
    rad_vel = ax.scatter(df_plot['ra_deg'], df_plot['dec_deg'],
                             s=20.0, 
                             c=df_plot['mean_kms'],#-df_plot['V_mean'], 
                             cmap='RdBu_r',
                             vmin=100,
                             vmax=500,
                         alpha = 0.5
                        )
    ax.invert_xaxis()
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    #ax.set_title(r'H$\alpha$ - H$\beta$')
    fig.colorbar(rad_vel, ax=ax, label='km/s')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)

jitter_ra = 0.000   # degrees
jitter_dec = 0.000  # degrees

ra_jittered = df_plot["ra_deg"] + rng.normal(0, jitter_ra, size=len(df_plot))
dec_jittered = df_plot["dec_deg"] + rng.normal(0, jitter_dec, size=len(df_plot))

cvals = df_plot["mean_kms"] - VELOCITY_REFERENCE_KMS

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=75.0,
        c=cvals,
   #     cmap="RdBu_r",
        cmap="gist_rainbow_r",
        vmin=-35,
        vmax=+35,
        marker = 's'
    )

    ax.invert_xaxis()
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")

    fig.colorbar(rad_vel, ax=ax, label="km/s")
    plt.show()

In [ ]:
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

# Center the display WCS on the sample; no online name lookup
c0 = SkyCoord(
    ra=float(df_validation_summary["ra_deg"].median()) * u.deg,
    dec=float(df_validation_summary["dec_deg"].median()) * u.deg,
)

wcs = WCS(naxis=2)
wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"]
wcs.wcs.cunit = ["deg", "deg"]

# Reference sky coordinate
wcs.wcs.crval = [c0.ra.deg, c0.dec.deg]

# Reference pixel
wcs.wcs.crpix = [500, 500]

# Pixel scale in degrees/pixel
# Negative RA scale gives standard astronomical orientation
wcs.wcs.cdelt = [-0.00005, 0.00005]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.visualization.wcsaxes import WCSAxes

rng = np.random.default_rng(42)

jitter_ra = 0.000
jitter_dec = 0.000

ra_jittered = df_plot["ra_deg"].to_numpy() + rng.normal(
    0, jitter_ra, size=len(df_plot)
)

dec_jittered = df_plot["dec_deg"].to_numpy() + rng.normal(
    0, jitter_dec, size=len(df_plot)
)

cvals = df_plot["mean_kms"].to_numpy() - VELOCITY_REFERENCE_KMS

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig = plt.figure(figsize=(7, 6))

    # Explicit WCSAxes construction
    ax = WCSAxes(
        fig,
        [0.12, 0.12, 0.72, 0.78],   # [left, bottom, width, height]
        wcs=wcs,
    )

    fig.add_axes(ax)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=15.0,
        c=cvals,
        cmap="RdBu_r",
        vmin=-45,
        vmax=+45,
        marker=".",
        transform=ax.get_transform("world"),
    )

    ax.coords[0].set_axislabel("Right Ascension")
    ax.coords[1].set_axislabel("Declination")

    ax.coords[0].set_major_formatter("hh:mm:ss")
    ax.coords[1].set_major_formatter("dd:mm:ss")

    ax.grid(color="0.55", alpha=0.6, linestyle="--")

    fig.colorbar(
        rad_vel,
        ax=ax,
        label=r"$v_{\rm comp} - VELOCITY_REFERENCE_KMS$ km s$^{-1}$",
    )

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.visualization.wcsaxes import WCSAxes

rng = np.random.default_rng(42)

jitter_ra = 0.000
jitter_dec = 0.000

ra_jittered = df_plot["ra_deg"].to_numpy() + rng.normal(
    0, jitter_ra, size=len(df_plot)
)

dec_jittered = df_plot["dec_deg"].to_numpy() + rng.normal(
    0, jitter_dec, size=len(df_plot)
)

cvals = df_plot["mean_kms"].to_numpy() - VELOCITY_REFERENCE_KMS

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig = plt.figure(figsize=(7, 6))

    # Explicit WCSAxes construction
    ax = WCSAxes(
        fig,
        [0.12, 0.12, 0.72, 0.78],   # [left, bottom, width, height]
        wcs=wcs,
    )

    fig.add_axes(ax)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=15.0,
        c=cvals,
        cmap="gist_rainbow_r",
        vmin=-35,
        vmax=+35,
        marker=".",
        transform=ax.get_transform("world"),
    )

    ax.coords[0].set_axislabel("Right Ascension")
    ax.coords[1].set_axislabel("Declination")

    ax.coords[0].set_major_formatter("hh:mm:ss")
    ax.coords[1].set_major_formatter("dd:mm:ss")

    ax.grid(color="0.55", alpha=0.6, linestyle="--")

    fig.colorbar(
        rad_vel,
        ax=ax,
        label=r"$v_{\rm comp} - VELOCITY_REFERENCE_KMS$ km s$^{-1}$",
    )

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.visualization.wcsaxes import WCSAxes

rng = np.random.default_rng(42)

jitter_ra = 0.000
jitter_dec = 0.000

ra_jittered = df_plot["ra_deg"].to_numpy() + rng.normal(
    0, jitter_ra, size=len(df_plot)
)

dec_jittered = df_plot["dec_deg"].to_numpy() + rng.normal(
    0, jitter_dec, size=len(df_plot)
)

cvals = df_plot["sigma_kms"].to_numpy()

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig = plt.figure(figsize=(7, 6))

    # Explicit WCSAxes construction
    ax = WCSAxes(
        fig,
        [0.12, 0.12, 0.72, 0.78],   # [left, bottom, width, height]
        wcs=wcs,
    )

    fig.add_axes(ax)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=15.0,
        c=cvals,
        cmap="gist_rainbow_r",
        vmin=5,
        vmax= 45,
        marker=".",
        transform=ax.get_transform("world"),
    )

    ax.coords[0].set_axislabel("Right Ascension")
    ax.coords[1].set_axislabel("Declination")

    ax.coords[0].set_major_formatter("hh:mm:ss")
    ax.coords[1].set_major_formatter("dd:mm:ss")

    ax.grid(color="0.55", alpha=0.6, linestyle="--")

    fig.colorbar(
        rad_vel,
        ax=ax,
        label=r"$\sigma$",
    )

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)
    rad_vel = ax.scatter(df_plot['ra_deg'], df_plot['dec_deg'],
                             s=15.0, 
                             c=np.log10(df_plot['amp']),#-df_plot['V_mean'], 
                             cmap='magma',
                           #  vmin=100,
                           #  vmax=500,
                         alpha = 1.0,
                         marker = '.'
                        )
    ax.invert_xaxis()
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    #ax.set_title(r'H$\alpha$ - H$\beta$')
    fig.colorbar(rad_vel, ax=ax, label='log10 peak amplitude (input flux units)')
    plt.show()

## Export
Tables retain identifiers and selection status. The JSON keeps the original five-column layout but uses only representatives with finite coordinates, velocity, amplitude and corrected dispersion. It excludes unresolved widths; the full representative table preserves them. All outputs are isolated in the `brightest_component` subfolder.

In [ ]:
import json

export_dir = out_dir / "brightest_component"
export_dir.mkdir(parents=True, exist_ok=True)
for name, table in {
    "df_validation_summary": df_validation_summary,
    "df_components": df_components,
    "df_representative": df_representative,
}.items():
    table.to_pickle(export_dir / f"{name}.pkl")
    table.to_csv(export_dir / f"{name}.csv", index=False)

export_values = np.column_stack([
    df_plot["ra_deg"], df_plot["dec_deg"],
    df_plot["mean_kms"] - VELOCITY_REFERENCE_KMS,
    np.log10(df_plot["amp"]), df_plot["sigma_kms"],
]).astype(float)
export_mask = np.isfinite(export_values).all(axis=1)
name = "FLAMES-30Dor-6563"
with open(export_dir / f"{name}-brightest-l.json", "w") as f:
    json.dump({"name": name, "data": export_values[export_mask].tolist()}, f, indent=3, allow_nan=False)
# Companion identities preserve the mapping to rows of the JSON.
df_plot.loc[export_mask, ["spectrum_index", "filename", "component_index"]].to_csv(
    export_dir / f"{name}-brightest-identities.csv", index=False
)
with open(export_dir / "representative_config.json", "w") as f:
    json.dump({
        "input_file": str(FILENAME_DATA), "decomposition_file": str(FILENAME_DATA_DECOMP),
        "selection": "largest finite positive fitted peak amplitude; ties lowest component_index",
        "lambda0_nm": LAMBDA0_NM, "velocity_reference_kms": VELOCITY_REFERENCE_KMS,
        "broadening_terms_kms": [BROADENING_1_KMS, BROADENING_2_KMS],
        "fwhm_to_sigma": FWHM_TO_SIGMA,
        "json_columns": ["ra_deg", "dec_deg", "relative_velocity_kms", "log10_amp", "sigma_kms"],
    }, f, indent=2)
print(f"Saved tables and representative JSON to: {export_dir}")
print(f"JSON rows: {export_mask.sum()}; omitted for nonfinite values: {(~export_mask).sum()}")